<a href="https://colab.research.google.com/github/aszczi/Urban_mobility_in_Cracow/blob/main/Opoznienia_KMK.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/aszczi/Urban_mobility_in_Cracow/blob/main/Opoznienia_KMK.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Analiza Opóźnień Komunikacji Miejskiej w Krakowie (GTFS-RT)

Podprojekt analizy rzeczywistych opóźnień komunikacji miejskiej w Krakowie. Dane pobierane są w czasie rzeczywistym z usług [GTFS-RT ZTP Kraków](https://gtfs.ztp.krakow.pl/).

Wykorzystujemy:
- **TripUpdates** (format `.pb` - Protobuf), aby pozyskać estymowane czasy przyjazdów i porównać je do planowanych.
- **Dane statyczne (GTFS)** do podpięcia lokalizacji geo (przystanków) oraz nazw linii.


*Dane pokazują tylko aktualne zmiany, a nie trend historyczny. Opóźnienia z systemu mogą różnić się od rzeczywistych.*

*Wyniki programu będą się znacząco róźnić w zależności od pory dnia.*

In [ ]:
import sys
import subprocess
import importlib
import warnings
import datetime
import os
import re
import time
import io
import zipfile
import urllib.request


def ensure_import(import_name, package_name=None):
    """Importuje pakiet, a gdy go brakuje – doinstalowuje go przez pip."""
    package_name = package_name or import_name
    try:
        return importlib.import_module(import_name)
    except ModuleNotFoundError:
        print(f"Instaluję brakujący pakiet: {package_name}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package_name])
        return importlib.import_module(import_name)

ensure_import("requests")
ensure_import("pandas")
ensure_import("plotly")
ensure_import("folium")
ensure_import("matplotlib")
ensure_import("networkx")
ensure_import("osmnx")
ensure_import("scipy")
ensure_import("google.transit.gtfs_realtime_pb2", "gtfs-realtime-bindings")

import requests
from google.transit import gtfs_realtime_pb2
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import folium
from folium.plugins import TimestampedGeoJson
import osmnx as ox
import networkx as nx
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
from IPython.display import display

warnings.filterwarnings("ignore")

LOCAL_TZ = "Europe/Warsaw"
pd.set_option("display.max_columns", 100)

print("Środowisko gotowe")


In [ ]:
import requests
import time
import pandas as pd
from google.transit import gtfs_realtime_pb2

LOCAL_TZ = "Europe/Warsaw"

def _safe_has_field(proto_message, field_name):
    try:
        return proto_message.HasField(field_name)
    except:
        return False

def fetch_gtfs_rt_delays():
    urls = {
        "Tramwaje": "https://gtfs.ztp.krakow.pl/TripUpdates_T.pb",
        "Autobusy": "https://gtfs.ztp.krakow.pl/TripUpdates_A.pb",
    }
    rows = []
    for v_type, url in urls.items():
        try:
            response = requests.get(url, timeout=30)
            if not response.content: continue
            feed = gtfs_realtime_pb2.FeedMessage()
            feed.ParseFromString(response.content)

            count_type = 0
            for entity in feed.entity:
                if not entity.HasField("trip_update"): continue
                tu = entity.trip_update


                raw_route = str(tu.trip.route_id or "")
                raw_trip = str(tu.trip.trip_id or "")
                line_key = raw_route if raw_route else raw_trip

                for stu in tu.stop_time_update:
                    delay = None
                    time_val = None
                    if _safe_has_field(stu, "arrival"):
                        delay = stu.arrival.delay
                        time_val = stu.arrival.time
                    elif _safe_has_field(stu, "departure"):
                        delay = stu.departure.delay
                        time_val = stu.departure.time

                    if delay is not None:
                        d_min = float(delay) / 60.0

                        if -5 <= d_min <= 30:
                            count_type += 1
                            rows.append({
                                "typ": v_type,
                                "line_num": line_key,
                                "stop_id": str(stu.stop_id),
                                "delay_sec": float(delay),
                                "delay_min": d_min,
                                "time": time_val
                            })
            print(f"[{v_type}] Znaleziono {count_type} rekordów w feedzie RT.")
        except Exception as e:
            print(f"Błąd połączenia z {v_type}: {e}")

    df = pd.DataFrame(rows)
    if not df.empty:
        dt = pd.to_datetime(df["time"], unit="s", utc=True, errors="coerce")
        df["datetime"] = dt.dt.tz_convert(LOCAL_TZ)
        df["hour"] = df["datetime"].dt.strftime("%H:00")
    return df.reset_index(drop=True)

df_delays_raw = fetch_gtfs_rt_delays()
df_delays = df_delays_raw[df_delays_raw["delay_min"] > 0].copy() if not df_delays_raw.empty else pd.DataFrame()

if not df_delays.empty:
    print(f"Pobrano {len(df_delays)} rekordów z dodatnim opóźnieniem.")
    display(df_delays.groupby('typ').size().rename('Liczba rekordów'))
else:
    print("Brak opóźnień w tej chwili (wszystko o czasie lub brak danych).")

In [ ]:
import re
import urllib.request
import zipfile
import io
import pandas as pd

def normalize_id(value):
    if pd.isna(value): return None
    s = str(value).strip()
    match = re.findall(r'(\d+)', s)
    return match[-1] if match else s

def fetch_current_gtfs_static():
    urls = {"Tramwaje": "https://gtfs.ztp.krakow.pl/GTFS_KRK_T.zip", "Autobusy": "https://gtfs.ztp.krakow.pl/GTFS_KRK_A.zip"}
    all_stops, all_routes = [], []
    for v_type, url in urls.items():
        try:
            req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(req, timeout=30) as resp:
                with zipfile.ZipFile(io.BytesIO(resp.read())) as z:
                    with z.open("stops.txt") as f:
                        s = pd.read_csv(f, dtype=str)
                        s["typ"] = v_type
                        all_stops.append(s)
                    with z.open("routes.txt") as f:
                        r = pd.read_csv(f, dtype=str)
                        r["typ"] = v_type
                        all_routes.append(r)
        except: pass
    return pd.concat(all_stops) if all_stops else pd.DataFrame(), pd.concat(all_routes) if all_routes else pd.DataFrame()

def merge_delays_with_static(df_delays, df_stops, df_routes):
    if df_delays.empty or df_stops.empty: return pd.DataFrame(), pd.DataFrame()
    d = df_delays.copy()
    d["merge_key"] = d["stop_id"].apply(normalize_id)

    s = df_stops.copy()
    s["merge_key"] = s["stop_id"].apply(normalize_id)


    merged = d.merge(s, on=["merge_key", "typ"], suffixes=("_rt", "_static"))


    r = df_routes.copy()
    r["merge_line"] = r["route_id"].apply(normalize_id)


    merged["merge_line"] = merged["line_num"].apply(normalize_id)

    final = merged.merge(r[["merge_line", "typ", "route_short_name"]], on=["merge_line", "typ"], how="left")
    final["linia"] = final["route_short_name"].fillna(final["merge_line"])

    for col in ["stop_lat", "stop_lon"]:
        final[col] = pd.to_numeric(final[col], errors="coerce")

    agg = final.groupby(["stop_name", "stop_lat", "stop_lon", "typ"], as_index=False).agg(
        mean_delay_min=("delay_min", "mean"),
        measurements_count=("delay_min", "count")
    )
    return final, agg

if 'df_delays_raw' in locals() and not df_delays_raw.empty:

    df_active = df_delays_raw[df_delays_raw['delay_min'] >= 0].copy()
    df_stops, df_routes = fetch_current_gtfs_static()
    df_merged, df_stops_delays = merge_delays_with_static(df_active, df_stops, df_routes)

    if not df_merged.empty:
        print(f"Sukces merge: {len(df_merged)} rekordów.")
        display(df_merged.groupby('typ').size().rename('Poprawnie dopasowane'))
    else:
        print("Błąd: Nie udało się dopasować rekordów. Sprawdź czy dane RT i statyczne są spójne.")

## Interaktywna mapa ruchu komunikacji miejskiej
Mapa przedstawia opóźnienia komunikacji na danej drodze.


Nakładamy dane z przystanków na mapę i animujemy je z użyciem GeoJSON. Używamy OSMnx do znalezienia najbliższych dróg.

Kolory przedstawiają wielkość opóźnienia na drodze.

In [ ]:
def _delay_to_color(delay, vmin=0, vmax=30):
    cmap = plt.get_cmap("RdYlGn_r")
    norm = mcolors.Normalize(vmin=vmin, vmax=vmax)
    return mcolors.to_hex(cmap(norm(max(vmin, min(float(delay), vmax)))))


def _static_stop_map(df_stops_plot):
    """Awaryjna mapa punktowa – działa nawet wtedy, gdy OSMnx/Overpass nie pobierze siatki ulic."""
    m = folium.Map(location=[50.0614, 19.9383], zoom_start=12, tiles="cartodbpositron")

    for _, row in df_stops_plot.iterrows():
        delay = float(row["mean_delay_min"])
        radius = max(5, min(20, 5 + delay / 2))
        folium.CircleMarker(
            location=[row["stop_lat"], row["stop_lon"]],
            radius=radius,
            color=_delay_to_color(delay),
            fill=True,
            fill_opacity=0.75,
            popup=(
                f"<b>{row['stop_name']}</b><br>"
                f"Typ: {row['typ']}<br>"
                f"Śr. opóźnienie: {delay:.1f} min<br>"
                f"Pomiarów: {int(row['measurements_count'])}"
            ),
        ).add_to(m)
    return m


if "df_merged" in locals() and not df_merged.empty:
    df_map = df_merged.dropna(subset=["stop_name", "stop_lat", "stop_lon", "hour"]).copy()
    df_map["stop_lat"] = pd.to_numeric(df_map["stop_lat"], errors="coerce")
    df_map["stop_lon"] = pd.to_numeric(df_map["stop_lon"], errors="coerce")
    df_map["delay_min"] = pd.to_numeric(df_map["delay_min"], errors="coerce")
    df_map = df_map.dropna(subset=["stop_lat", "stop_lon", "delay_min"])

    if df_map.empty:
        print("Brak poprawnych współrzędnych do mapy.")
    else:

        df_stop_hour = (
            df_map.groupby(["stop_name", "stop_lat", "stop_lon", "typ"], as_index=False)
            .agg(mean_delay_min=("delay_min", "mean"), measurements_count=("delay_min", "count"))

        )

        folium_map = folium.Map(location=[50.0614, 19.9383], zoom_start=13, tiles="cartodbdark_matter")
        try:
            lokalizacja = "Kraków, Poland"
            print(f"Pobieranie geometrii dróg dla: {lokalizacja}...")
            G = ox.graph_from_place(lokalizacja, network_type="drive", simplify=True)

            unique_points = df_stop_hour.drop_duplicates(subset=["stop_name", "stop_lat", "stop_lon"]).reset_index(drop=True)
            unique_points["_point_id"] = range(len(unique_points))

            print("Przypinanie przystanków do najbliższych ulic...")
            try:
                nearest_edges = ox.nearest_edges(G, X=unique_points["stop_lon"].values, Y=unique_points["stop_lat"].values)
            except AttributeError:
                nearest_edges = ox.distance.nearest_edges(G, X=unique_points["stop_lon"].values, Y=unique_points["stop_lat"].values)

            point_to_edge = dict(zip(unique_points["_point_id"], nearest_edges))
            df_stop_hour = df_stop_hour.merge(
                unique_points[["_point_id", "stop_name", "stop_lat", "stop_lon"]],
                on=["stop_name", "stop_lat", "stop_lon"],
                how="left",
            )

            print("Budowanie animowanej warstwy GeoJSON...")
            for _, row in df_stop_hour.iterrows():
                if pd.isna(row["_point_id"]):
                    continue
                edge = point_to_edge.get(int(row["_point_id"]))
                if edge is None:
                    continue

                if len(edge) == 3:
                    u, v, key = edge
                    edge_data = G.get_edge_data(u, v, key) or {}
                else:
                    u, v = edge[:2]
                    data = G.get_edge_data(u, v) or {}
                    edge_data = next(iter(data.values())) if isinstance(data, dict) and data and "geometry" not in data else data

                if edge_data and "geometry" in edge_data:
                    coords = [[x, y] for x, y in edge_data["geometry"].coords]
                else:
                    coords = [[G.nodes[u]["x"], G.nodes[u]["y"]], [G.nodes[v]["x"], G.nodes[v]["y"]]]

                delay = float(row["mean_delay_min"])
                folium_coords = [[y, x] for x, y in coords]
                folium.PolyLine(
                    locations=folium_coords,
                    color=_delay_to_color(delay),
                    weight=7,
                    opacity=0.85,
                    tooltip=f"{row['stop_name']} – {delay:.1f} min"
                ).add_to(folium_map)

            if True:
                display(folium_map)
            else:
                print("Nie zbudowano warstw ulic – pokazuję mapę punktową przystanków.")
                display(_static_stop_map(df_stops_delays))

        except Exception as exc:
            print(f"OSMnx/Overpass nie zwrócił siatki ulic ({exc}). Pokazuję awaryjną mapę punktową.")
            display(_static_stop_map(df_stops_delays))
else:
    print("Brak danych z koordynatami (df_merged) do wyświetlenia mapy.")


## Wykresy (Gdzie są największe opóźnienia)
Przeanalizujmy, które przystanki oraz które linie notują średnio największe opóźnienia w pozyskanej próbce czasowej.

In [ ]:
import plotly.express as px

def show_bar_min(df, x_col, y_col, color_val, title, label_dict):
    if df is None or df.empty:
        print(f"Brak danych do wyświetlenia dla: {title}")
        return
    fig = px.bar(df, x=x_col, y=y_col, title=title, labels=label_dict,
                 color_discrete_sequence=[color_val], text_auto='.1f')
    fig.update_layout(xaxis_tickangle=-45)
    fig.show()

if 'df_merged' in locals() and not df_merged.empty:
    # 1. Top opóźnienia przystanków autobusowych
    bus_stops = df_stops_delays[df_stops_delays['typ'] == 'Autobusy'].sort_values('mean_delay_min', ascending=False).head(15)
    show_bar_min(bus_stops, 'stop_name', 'mean_delay_min', '#ff7f0e',
                 "1. Top 15: Opóźnienia przystanków autobusowych [min]",
                 {"stop_name": "Przystanek", "mean_delay_min": "Śr. opóźnienie (min)"})

    # 2. Top spóźniające się linie autobusowe
    bus_routes = df_merged[df_merged['typ'] == 'Autobusy'].groupby('linia')['delay_min'].mean().reset_index()
    bus_routes = bus_routes.sort_values('delay_min', ascending=False).head(15)
    show_bar_min(bus_routes, 'linia', 'delay_min', '#ff7f0e',
                 "2. Top 15: Spóźniające się linie autobusowe [min]",
                 {"linia": "Linia", "delay_min": "Śr. opóźnienie (min)"})

    # 3. Top opóźnienia przystanków tramwajowych
    tram_stops_all = df_stops_delays[df_stops_delays['typ'] == 'Tramwaje']
    tram_stops = tram_stops_all.sort_values('mean_delay_min', ascending=False).head(15)
    show_bar_min(tram_stops, 'stop_name', 'mean_delay_min', '#1f77b4',
                 "3. Top 15: Opóźnienia przystanków tramwajowych [min]",
                 {"stop_name": "Przystanek", "mean_delay_min": "Śr. opóźnienie (min)"})

    # 4. Top spóźniające się linie tramwajowe
    tram_routes_all = df_merged[df_merged['typ'] == 'Tramwaje'].groupby('linia')['delay_min'].mean().reset_index()
    tram_routes = tram_routes_all.sort_values('delay_min', ascending=False).head(15)
    show_bar_min(tram_routes, 'linia', 'delay_min', '#1f77b4',
                 "4. Top 15: Spóźniające się linie tramwajowe [min]",
                 {"linia": "Linia", "delay_min": "Śr. opóźnienie (min)"})

    if not tram_routes_all.empty and tram_routes_all['delay_min'].max() == 0:
        print("\nINFO: Wszystkie zarejestrowane tramwaje kursują obecnie punktualnie (opóźnienie 0.0).")
else:
    print("Brak zmergowanych danych. Uruchom wcześniejsze komórki.")